In [1]:
import pandas as pd

In [2]:
class Author:
    def __init__(self, name, paper_id, coauthors, df):
        self.name = name
#         self.coauthor = coauthors
        self.paper_id = paper_id
        filtered_df = df.loc[(int(paper_id), name)]

        self.depart = ast.literal_eval(filtered_df['department'].values[0])
        if coauthors and isinstance(coauthors, str): 
            self.coauthors = ast.literal_eval(coauthors)
        else:
            self.coauthors = []  # 或者设置为其他默认值

#         if str(filtered_df['coauthors'].values[0]).lower() == 'nan': 
#             self.coauthors = []
#         else:
#             self.coauthors = ast.literal_eval(filtered_df['coauthors'].values[0])
        
        self.title = filtered_df['title'].values[0]
        self.abstract = filtered_df['abstract'].values[0]
        self.keywords = filtered_df['keywords'].values[0]
        self.year = filtered_df['publish_time'].values[0]

In [3]:
from tqdm import tqdm
import pandas as pd
import csv 
import ast
import string

same_name_authors = {}
records = set()
df = pd.read_csv('../DisambiguationData/Authorship_English.csv')


df.set_index(['paper id', 'author_name', 'coauthors'], inplace=True)
with open('../DisambiguationData/Authorship_English.csv', 'r') as f:
    for i, row in tqdm(enumerate(csv.reader(f))):
        if i != 0:
            if (row[2], row[0], row[4]) not in records:
#                 print(row[2], row[0], row[4])
#                 if row[4] and isinstance(row[4], str): 
#                     coauthors = ast.literal_eval(row[4])
#                 else:
#                     coauthors = []  # 或者设置为其他默认值

                author = Author(row[2], row[0], row[4], df)
                if row[2] not in same_name_authors.keys():
                    same_name_authors[row[2]] = [author]
                else:
                    same_name_authors[row[2]].append(author)
                records.add((row[2], row[0], row[4]))

0it [00:00, ?it/s]/var/folders/bw/w2pkl8js06ngvf9n2yn4j_tr0000gn/T/ipykernel_16204/3421359664.py:6: PerformanceWarning: indexing past lexsort depth may impact performance.
  filtered_df = df.loc[(int(paper_id), name)]
247050it [00:59, 4138.49it/s]


In [4]:
len(same_name_authors)

68601

## 1. use coauthors

In [5]:
def _same_name_coauthors(author0, author1):
    """
    return True when they are same
    """
    if author0.paper_id != author1.paper_id:
        if len(set(author0.coauthors) & set(author1.coauthors)) > 0:
            return True
    
    return False

In [6]:
import numpy as np
with open("../DisambiguationData/English_names_disambiguation/1_author_doi_coauthors.csv", 'w', encoding='utf-8') as fw:
    csv_write = csv.writer(fw)
    csv_head = ["author_index", "author_name", "paper_index", 'year', 'title', 'abstract', 'keywords']
    csv_write.writerow(csv_head)
    name_index = 0
    count = 0
    for name in tqdm(same_name_authors):
        author_length = len(same_name_authors[name])
        if author_length > 1:
            indexes = np.arange(author_length)
            group_indexes = []
            while indexes.size != 0:
                current_group_index = list()
                current_group_index.append(indexes[0])
                next_indexes = []
                for i in indexes[1:]:
                    author0 = same_name_authors[name][indexes[0]]
                    author1 = same_name_authors[name][i]
                    if _same_name_coauthors(author0, author1):
                        current_group_index.append(i)
                    else:
                        next_indexes.append(i)
                m = len(current_group_index)
                n = 1
                while m > n:
                    f = m
                    for q in range(m-1, n-1, -1):
                        remove_indexs = []
                        for j in next_indexes:
                            author0 = same_name_authors[name][current_group_index[q]]
                            author1 = same_name_authors[name][j]
                            if _same_name_coauthors(author0, author1):
                                current_group_index.append(j)
#                                 next_indexes.remove(j)
                                remove_indexs.append(j)
                            else:
                                 pass
                        next_indexes = [q for q in next_indexes if q not in set(remove_indexs)]
                
                    m = len(current_group_index)
                    n = f
                group_indexes.append(np.array(current_group_index))
                indexes = np.array(next_indexes)
            for i in range(len(group_indexes)):
                for index in group_indexes[i]:
                    author = same_name_authors[name][index]
                    csv_write.writerow([name_index + i, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + len(group_indexes)
        else:
            author = same_name_authors[name][0]
            csv_write.writerow([name_index, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + 1
        count += 1


100%|████████████████████████████████████████████████| 68601/68601 [00:05<00:00, 12058.84it/s]


## 2. use affiliation

In [7]:
from Levenshtein import distance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def is_same_department(affiliation1, affiliation2, threshold=0.6):
    # Calculate the Levenshtein distance between the affiliations
    if affiliation1 != [] and affiliation2 != []:
        similarity_ratio_lists = set()
        for i in affiliation1:
            for j in affiliation2:
#                 vectorizer = TfidfVectorizer().fit_transform([i, j])

#                 # 计算余弦相似度
#                 similarity_ratio = cosine_similarity(vectorizer)[0, 1]

                dist = distance(i, j)

                # Calculate the similarity ratio
                max_len = max(len(i), len(j))
                similarity_ratio = 1 - (dist / max_len)
                similarity_ratio_lists.add(similarity_ratio)
        # Check if the similarity ratio is above the threshold

        if max(similarity_ratio_lists) >= threshold:
            return True
        else:
            for str1 in affiliation1:
                preprocessed_str1 = re.sub(r'\W+', '', str1)
                for str2 in affiliation2:
                    preprocessed_str2 = re.sub(r'\W+', '', str2)
                    # 检查预处理后的字符串是否互相包含
                    if preprocessed_str1 in preprocessed_str2 or preprocessed_str2 in preprocessed_str1:
                        return True

            return False
    else:
        return False
    
def _same_name_coauthors_affiliation(author0, author1):
    """
    return True when they are same
    """
    if author0.paper_id != author1.paper_id:
        if len(set(author0.coauthors) & set(author1.coauthors)) > 0:
            return True
    #     tfidf_vectorizer = TfidfVectorizer()
    #     author0_tfidf = tfidf_vectorizer.fit_transform(author0.depart)
    #     author1_tfidf = tfidf_vectorizer.transform(author1.depart)
        if is_same_department(author0.depart, author1.depart):
            return True
        # if len(set(author0.affiliation) & set(author1.affiliation)) > 0:
        #     return True
    

    return False

In [8]:
import numpy as np
import re

with open("../DisambiguationData/English_names_disambiguation/2_author_doi_affiliation.csv", 'w', encoding='utf-8') as fw:
    csv_write = csv.writer(fw)
    csv_head = ["author_index", "author_name", "paper_index", 'year', 'title', 'abstract', 'keywords']
    csv_write.writerow(csv_head)
    name_index = 0
    count = 0
    for name in tqdm(same_name_authors):
        author_length = len(same_name_authors[name])
        if author_length > 1:
            indexes = np.arange(author_length)
            group_indexes = []
            while indexes.size != 0:
                current_group_index = list()
                current_group_index.append(indexes[0])
                next_indexes = []
                for i in indexes[1:]:
                    author0 = same_name_authors[name][indexes[0]]
                    author1 = same_name_authors[name][i]
                    if _same_name_coauthors_affiliation(author0, author1):
                        current_group_index.append(i)
                    else:
                        next_indexes.append(i)
                m = len(current_group_index)
                n = 1
                while m > n:
                    f = m
                    for q in range(m-1, n-1, -1):
                        remove_indexs = []
                        for j in next_indexes:
                            author0 = same_name_authors[name][current_group_index[q]]
                            author1 = same_name_authors[name][j]
                            if _same_name_coauthors_affiliation(author0, author1):
                                current_group_index.append(j)
                                remove_indexs.append(j)
                            else:
                                 pass
                        next_indexes = [q for q in next_indexes if q not in set(remove_indexs)]
                    m = len(current_group_index)
                    n = f
                group_indexes.append(np.array(current_group_index))
                indexes = np.array(next_indexes)
            for i in range(len(group_indexes)):
                for index in group_indexes[i]:
                    author = same_name_authors[name][index]
                    csv_write.writerow([name_index + i, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + len(group_indexes)
        else:
            author = same_name_authors[name][0]
            csv_write.writerow([name_index, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + 1
        count += 1



100%|█████████████████████████████████████████████████| 68601/68601 [00:13<00:00, 4926.46it/s]


## 3. Add paper infos(title)

In [9]:
simarility_titles = pd.read_csv("../DisambiguationData/AbstractSimilarity.csv")
# 建立哈希映射
mapping = { (min(row.paper_index1, row.paper_index2), max(row.paper_index1, row.paper_index2)): row.similarity_en 
            for _, row in simarility_titles.iterrows() }

In [10]:
def calculate_similarity_en(pid1, pid2):
    key = (min(pid1, pid2), max(pid1, pid2))
    similarity_en = mapping.get(key)

    return similarity_en

In [11]:
def _same_name_with_title(author0, author1):
    """
    return True when they are same
    """
    if author0.paper_id != author1.paper_id:
        if len(set(author0.coauthors) & set(author1.coauthors)) > 0:
            return True
    #     tfidf_vectorizer = TfidfVectorizer()
    #     author0_tfidf = tfidf_vectorizer.fit_transform(author0.depart)
    #     author1_tfidf = tfidf_vectorizer.transform(author1.depart)
        if is_same_department(author0.depart, author1.depart):
            return True
        # if len(set(author0.affiliation) & set(author1.affiliation)) > 0:
        #     return True

    #     if calculate_similarity_en(author0.abstract, author1.abstract, model) > 0.9:
    #         return True
        if calculate_similarity_en(int(author0.paper_id), int(author1.paper_id)) >= 0.9620689655172414:

            return True
    
    return False

In [12]:
import numpy as np
with open("../DisambiguationData/English_names_disambiguation/3_author_doi_title.csv", 'w', encoding='utf-8') as fw:
    csv_write = csv.writer(fw)
    csv_head = ["author_index", "author_name", "paper_index", 'year', 'title', 'abstract', 'keywords']
    csv_write.writerow(csv_head)
    name_index = 0
    count = 0
    for name in tqdm(same_name_authors):
        author_length = len(same_name_authors[name])
        if author_length > 1:
            indexes = np.arange(author_length)
            group_indexes = []
            while indexes.size != 0:
                current_group_index = list()
                current_group_index.append(indexes[0])
                next_indexes = []
                for i in indexes[1:]:
                    author0 = same_name_authors[name][indexes[0]]
                    author1 = same_name_authors[name][i]
                    if _same_name_with_title(author0, author1):
                        current_group_index.append(i)
                    else:
                        next_indexes.append(i)
                m = len(current_group_index)
                n = 1
                while m > n:
                    f = m
                    for q in range(m-1, n-1, -1):
                        remove = []
                        for j in next_indexes:
                            author0 = same_name_authors[name][current_group_index[q]]
                            author1 = same_name_authors[name][j]
                            if _same_name_with_title(author0, author1):
                                current_group_index.append(j)
                                remove.append(j)
#                                 next_indexes.remove(j)
                            else:
                                 pass
                        next_indexes = [q for q in next_indexes if q not in set(remove)]
                    m = len(current_group_index)
                    n = f
                group_indexes.append(np.array(current_group_index))
                indexes = np.array(next_indexes)
            for i in range(len(group_indexes)):
                for index in group_indexes[i]:
                    author = same_name_authors[name][index]
                    csv_write.writerow([name_index + i, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + len(group_indexes)
        else:
            author = same_name_authors[name][0]
            csv_write.writerow([name_index, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + 1
        count += 1


100%|█████████████████████████████████████████████████| 68601/68601 [00:15<00:00, 4554.59it/s]


## 4. use citations

In [13]:
citations_info = pd.read_csv('../DisambiguationData/GET_citations/Processed_Citations.csv')[['Cited_Title','Citing_Title']]

In [14]:
Paper_Translation = pd.read_csv('../DisambiguationData/Paper_Translation.csv')
title_cn_en = dict(zip(Paper_Translation['Title'], Paper_Translation['Title_English']))

In [15]:
# 增加 Cited_Title_en 列，根据字典匹配翻译，如果没有匹配则保持原始标题
citations_info['Cited_Title_en'] = citations_info['Cited_Title'].apply(lambda x: title_cn_en.get(x, x))
citations_info['Citing_Title_en'] = citations_info['Citing_Title'].apply(lambda x: title_cn_en.get(x, x))

In [19]:
# 使用 groupby 对 DataFrame 进行分组，并将列 'B' 的值收集到列表中
result_df = citations_info.groupby('Cited_Title_en')['Citing_Title_en'].apply(list).reset_index()

# 将结果 DataFrame 转换为字典
citing_relation_en = dict(zip(result_df['Cited_Title_en'], result_df['Citing_Title_en']))

In [20]:
def _same_name_with_title_citation(author0, author1):
    """
    return True when they are same
    """
    if author0.paper_id != author1.paper_id:
        if len(set(author0.coauthors) & set(author1.coauthors)) > 0:
            return True
    #     tfidf_vectorizer = TfidfVectorizer()
    #     author0_tfidf = tfidf_vectorizer.fit_transform(author0.depart)
    #     author1_tfidf = tfidf_vectorizer.transform(author1.depart)
        if is_same_department(author0.depart, author1.depart):
            return True
        # if len(set(author0.affiliation) & set(author1.affiliation)) > 0:
        #     return True

        if calculate_similarity_en(int(author0.paper_id), int(author1.paper_id)) >= 0.9620689655172414:
            return True


        if author0.title in citing_relation_en:
            if author1.title in citing_relation_en[author0.title]:
                return True
        if author1.title in citing_relation_en:
            if author0.title in citing_relation_en[author1.title]:
                return True
    
    return False

In [21]:
import numpy as np
with open("../DisambiguationData/English_names_disambiguation/4_author_doi_citation.csv", 'w', encoding='utf-8') as fw:
    csv_write = csv.writer(fw)
    csv_head = ["author_index", "author_name", "paper_index", 'year', 'title', 'abstract', 'keywords']
    csv_write.writerow(csv_head)
    name_index = 0
    count = 0
    for name in tqdm(same_name_authors):
        author_length = len(same_name_authors[name])
        if author_length > 1:
            indexes = np.arange(author_length)
            group_indexes = []
            while indexes.size != 0:
                current_group_index = list()
                current_group_index.append(indexes[0])
                next_indexes = []
                for i in indexes[1:]:
                    author0 = same_name_authors[name][indexes[0]]
                    author1 = same_name_authors[name][i]
                    if _same_name_with_title_citation(author0, author1):
                        current_group_index.append(i)
                    else:
                        next_indexes.append(i)
                m = len(current_group_index)
                n = 1
                while m > n:
                    f = m
                    for q in range(m-1, n-1, -1):
                        remove_indexs = []
                        for j in next_indexes:
                            author0 = same_name_authors[name][current_group_index[q]]
                            author1 = same_name_authors[name][j]
                            if _same_name_with_title_citation(author0, author1):
                                current_group_index.append(j)
                                remove_indexs.append(j)
                            else:
                                 pass
                        next_indexes = [q for q in next_indexes if q not in set(remove_indexs)]
                    m = len(current_group_index)
                    n = f
                group_indexes.append(np.array(current_group_index))
                indexes = np.array(next_indexes)
            for i in range(len(group_indexes)):
                for index in group_indexes[i]:
                    author = same_name_authors[name][index]
                    csv_write.writerow([name_index + i, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + len(group_indexes)
        else:
            author = same_name_authors[name][0]
            csv_write.writerow([name_index, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + 1
        count += 1

100%|█████████████████████████████████████████████████| 68601/68601 [00:15<00:00, 4418.37it/s]


## 4.compare

In [22]:
def same_author_paper(df):
    # Assuming df is your DataFrame
    author_map = pd.Series(df['author_name'].values,index=df['author_index']).to_dict()

    # Finding keys with the same value
    value_to_keys_map = {}
    for key, value in author_map.items():
        value_to_keys_map.setdefault(value, []).append(key)

    # Filter out values that occur only once (unique author names)
    keys_with_duplicate_values = {author: keys for author, keys in value_to_keys_map.items()}
    author_doi = keys_with_duplicate_values
    length = [len(keys_with_duplicate_values[i]) for i in keys_with_duplicate_values]
    print('number of same names and same papers: ',length.count(1))
    print('number of same names and different papers: ',len(length) - length.count(1))

In [23]:
names_with_one_paper = []
for i in same_name_authors:
    if len(same_name_authors[i]) == 1:
        names_with_one_paper.append(i)
print('-----before------')
print('unqiue names: ', len(same_name_authors))
print('names with one paper: ', len(names_with_one_paper))
print('names with more than one paper: ', len(same_name_authors)- len(names_with_one_paper))

-----before------
unqiue names:  68601
names with one paper:  35160
names with more than one paper:  33441


In [24]:
df_uniqe = pd.read_csv('../DisambiguationData/English_names_disambiguation/1_author_doi_coauthors.csv')
length = same_author_paper(df_uniqe)

number of same names and same papers:  47519
number of same names and different papers:  21082


In [25]:
df_uniqe2 = pd.read_csv('../DisambiguationData/English_names_disambiguation/2_author_doi_affiliation.csv')
length = same_author_paper(df_uniqe2)

number of same names and same papers:  53799
number of same names and different papers:  14802


In [26]:
df_uniqe3 = pd.read_csv('../DisambiguationData/English_names_disambiguation/3_author_doi_title.csv')
length = same_author_paper(df_uniqe3)

number of same names and same papers:  54024
number of same names and different papers:  14577


In [27]:
df_uniqe4 = pd.read_csv('../DisambiguationData/English_names_disambiguation/4_author_doi_citation.csv')
length = same_author_paper(df_uniqe4)

number of same names and same papers:  54134
number of same names and different papers:  14467


# sinatra

In [87]:
from Levenshtein import distance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def is_same_department(affiliation1, affiliation2, threshold=0.6):
    # Calculate the Levenshtein distance between the affiliations
    if affiliation1 != [] and affiliation2 != []:
        similarity_ratio_lists = set()
        for i in affiliation1:
            for j in affiliation2:
#                 vectorizer = TfidfVectorizer().fit_transform([i, j])

#                 # 计算余弦相似度
#                 similarity_ratio = cosine_similarity(vectorizer)[0, 1]

                dist = distance(i, j)

                # Calculate the similarity ratio
                max_len = max(len(i), len(j))
                similarity_ratio = 1 - (dist / max_len)
                similarity_ratio_lists.add(similarity_ratio)
        # Check if the similarity ratio is above the threshold

        if max(similarity_ratio_lists) >= threshold:
            return True
        else:
            for str1 in affiliation1:
                preprocessed_str1 = re.sub(r'\W+', '', str1)
                for str2 in affiliation2:
                    preprocessed_str2 = re.sub(r'\W+', '', str2)
                    # 检查预处理后的字符串是否互相包含
                    if preprocessed_str1 in preprocessed_str2 or preprocessed_str2 in preprocessed_str1:
                        return True

            return False
    else:
        return False
    
def _same_name_coauthors_Sinatra(author0, author1):
    """
    return True when they are same
    """
    if author0.paper_id != author1.paper_id:
        if len(set(author0.coauthors) & set(author1.coauthors)) > 0:
            return True
    #     tfidf_vectorizer = TfidfVectorizer()
    #     author0_tfidf = tfidf_vectorizer.fit_transform(author0.depart)
    #     author1_tfidf = tfidf_vectorizer.transform(author1.depart)
        if is_same_department(author0.depart, author1.depart):
            return True
        # if len(set(author0.affiliation) & set(author1.affiliation)) > 0:
        #     return True
        if author0.title in citing_relation_en:
            if author1.title in citing_relation_en[author0.title]:
                return True
        if author1.title in citing_relation_en:
            if author0.title in citing_relation_en[author1.title]:
                return True


    return False

In [88]:
import numpy as np
import re
with open("../DisambiguationData/English_names_disambiguation/5_author_doi_Sinatra.csv", 'w', encoding='utf-8') as fw:
    csv_write = csv.writer(fw)
    csv_head = ["author_index", "author_name", "paper_index", 'year', 'title', 'abstract', 'keywords']
    csv_write.writerow(csv_head)
    name_index = 0
    count = 0
    for name in tqdm(same_name_authors):
        author_length = len(same_name_authors[name])
        if author_length > 1:
            indexes = np.arange(author_length)
            group_indexes = []
            while indexes.size != 0:
                current_group_index = list()
                current_group_index.append(indexes[0])
                next_indexes = []
                for i in indexes[1:]:
                    author0 = same_name_authors[name][indexes[0]]
                    author1 = same_name_authors[name][i]
                    if _same_name_coauthors_Sinatra(author0, author1):
                        current_group_index.append(i)
                    else:
                        next_indexes.append(i)
                m = len(current_group_index)
                n = 1
                while m > n:
                    f = m
                    for q in range(m-1, n-1, -1):
                        remove_indexs = []
                        for j in next_indexes:
                            author0 = same_name_authors[name][current_group_index[q]]
                            author1 = same_name_authors[name][j]
                            if _same_name_coauthors_Sinatra(author0, author1):
                                current_group_index.append(j)
                                remove_indexs.append(j)
                            else:
                                 pass
                        next_indexes = [q for q in next_indexes if q not in set(remove_indexs)]
                    m = len(current_group_index)
                    n = f
                group_indexes.append(np.array(current_group_index))
                indexes = np.array(next_indexes)
            for i in range(len(group_indexes)):
                for index in group_indexes[i]:
                    author = same_name_authors[name][index]
                    csv_write.writerow([name_index + i, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + len(group_indexes)
        else:
            author = same_name_authors[name][0]
            csv_write.writerow([name_index, author.name, author.paper_id, author.year, author.title, author.abstract, author.keywords])
            name_index = name_index + 1
        count += 1

100%|█████████████████████████████████████| 68620/68620 [00:14<00:00, 4658.49it/s]
